In [1]:
print("HELLO")

HELLO


In [5]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Document
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, GeoRadius, GeoPoint

from dotenv import load_dotenv
import os


load_dotenv()

QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")


client = QdrantClient(
    url="https://28f6622d-7880-4025-a574-e176193450eb.eu-central-1-0.aws.cloud.qdrant.io",
    api_key=QDRANT_API_KEY,
    cloud_inference=True
)

In [6]:
from qdrant_client.models import PayloadSchemaType

client.create_payload_index(
    collection_name="workers",
    field_name="location",
    field_schema=PayloadSchemaType.GEO,
)

UpdateResult(operation_id=12, status=<UpdateStatus.COMPLETED: 'completed'>)

In [17]:
from langchain_huggingface import HuggingFaceEmbeddings

job_desc = "Need a plumber for pipe fixing quick, close"
embedder = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7876.12it/s]


In [18]:
job_embedding = embedder.embed_query(job_desc)
job_long = 34
job_lat = 55.3

In [19]:
job_embedding

[-0.030573902651667595,
 -0.0657176747918129,
 0.09317468851804733,
 0.00019549472199287266,
 -0.09140744805335999,
 -0.04563360661268234,
 -0.04541980102658272,
 0.022649189457297325,
 -0.01502543967217207,
 -0.015198871493339539,
 -0.04782275855541229,
 -0.04035303741693497,
 -0.06122073531150818,
 -0.03028307482600212,
 -0.11918678134679794,
 0.0023243455216288567,
 -0.04206262156367302,
 0.0510433092713356,
 0.016859687864780426,
 -0.10918860882520676,
 -0.12914924323558807,
 0.08670279383659363,
 -0.0111460005864501,
 -0.05375925451517105,
 -0.02640146017074585,
 -0.026352528482675552,
 0.04558410868048668,
 0.021828100085258484,
 -0.015422720462083817,
 -0.03576626628637314,
 0.009871507994830608,
 -0.025897903367877007,
 -0.0398746132850647,
 -0.06019575893878937,
 0.04183503985404968,
 0.045897290110588074,
 0.04282865300774574,
 0.02658194862306118,
 0.01457404438406229,
 -0.006260125432163477,
 0.09732518345117569,
 -0.04884133115410805,
 0.007932557724416256,
 -0.05323022231

In [ ]:
results = client.query_points(
    collection_name="workers",
    query=job_embedding,          # not query_vector
    query_filter=Filter(
        must=[
            FieldCondition(
                key="location",
                geo_radius=GeoRadius(
                    center=GeoPoint(lon=job_long, lat=job_lat),
                    radius=1000000.0,
                ),
            )
        ]
    ),
    limit=50,
)

points = results.points

In [16]:
results

QueryResponse(points=[])

In [20]:
info = client.get_collection("workers")
print(info.points_count)

6


In [22]:
raw = client.query_points(
    collection_name="workers",
    query=job_embedding,
    limit=5,
)
for point in raw.points:
    print(point)

id='00d7ff07-0d6d-59eb-badc-437a6e249cef' version=3 score=0.58823967 payload={'name': 'Tejas', 'worker_id': '00d7ff07-0d6d-59eb-badc-437a6e249cef', 'skills': ['pipe repair'], 'tools': [], 'keywords': ['plumbing', 'pipe repair'], 'profession': 'Plumbing', 'geo_location': {'lon': 34.0, 'lat': 55.5}, 'score': 4.0} vector=None shard_key=None order_value=None
id='80a37e09-7831-549d-804e-012c1549d09c' version=4 score=0.29445195 payload={'name': 'Tanishq', 'worker_id': '80a37e09-7831-549d-804e-012c1549d09c', 'skills': ['cutting overgrown plants and bushes', 'pruning trees and shrubs (top-down pruning)', 'sharpening gardening tools', 'planning and scheduling garden jobs', 'client communication and expectation setting', 'time and cost estimation for gardening projects', 'plant care including watering, fertilizing, and seasonal planting', 'handling difficult plants such as neem trees', 'completing large-scale garden work in a single day'], 'tools': ['big scissors', 'sharpening tools'], 'keywords

In [23]:
scroll_result = client.scroll(collection_name="workers", limit=3, with_payload=True)
for point in scroll_result[0]:
    print(point.payload)

{'name': 'Tejas', 'worker_id': '00d7ff07-0d6d-59eb-badc-437a6e249cef', 'skills': ['pipe repair'], 'tools': [], 'keywords': ['plumbing', 'pipe repair'], 'profession': 'Plumbing', 'geo_location': {'lon': 34.0, 'lat': 55.5}, 'score': 4.0}
{'name': 'Tanishq', 'worker_id': '60c27311-9672-53f4-849b-83621525931c', 'skills': ['photography'], 'tools': [], 'keywords': ['divorce photography', 'sensitive photography', 'emotionally charged shoots', 'session planning', 'session management', 'final image delivery', 'client comfort', 'client relationship management', 'portrait photography', 'photo editing', 'image retouching', 'photography'], 'profession': 'photographer', 'geo_location': {'lon': 34.0, 'lat': 55.5}, 'score': 4.0}
{'name': 'Tanishq', 'worker_id': '6e8f3c5a-71f8-5e13-ba7b-8d0ed6e3ac22', 'skills': ['RAG pipeline development', 'data chunking', 'data normalization', 'JSON schema design', 'typecasting', 'dynamic device input handling', 'uniform data model creation'], 'tools': ['LangChain'], 

In [24]:
from qdrant_client.models import PayloadSchemaType

client.create_payload_index(
    collection_name="workers",
    field_name="location",
    field_schema=PayloadSchemaType.GEO,
)

client.create_payload_index(
    collection_name="workers",
    field_name="skills",
    field_schema=PayloadSchemaType.KEYWORD,  # enables MatchAny on list fields
)

UpdateResult(operation_id=16, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:

embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [35]:
radius_km = 20
top_k = 5

In [36]:
client.create_payload_index(
    collection_name="workers",
    field_name="geo_location",   # match actual payload key
    field_schema=PayloadSchemaType.GEO,
)

results = client.query_points(
    collection_name="workers",
    query=job_embedding,
    query_filter=Filter(
        must=[
            FieldCondition(
                key="geo_location",   # match actual payload key
                geo_radius=GeoRadius(
                    center=GeoPoint(lon=job_long, lat=job_lat),
                    radius=radius_km * 1000,
                ),
            ),
        ]
    ),
    limit=top_k,
    with_payload=True,
)

In [37]:
results

QueryResponse(points=[])

In [31]:
scroll_result = client.scroll(collection_name="workers", limit=10, with_payload=True)
for point in scroll_result[0]:
    print(point.payload.get("location"))

None
None
None
None
None
None
